# HW03 — Michelson Interferometers, Modulations, and Transfer Functions
Syracuse University — Lasers and Optomechanics

Works on paper. Let's see if it works in code.

In [ ]:
%matplotlib inline
from ipywidgets import *

import sympy as sp
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

sp.init_printing(use_latex='mathjax')

In [ ]:
plt.style.use('dark_background')

fontsize = 14
mpl.rcParams.update({
    'text.usetex': False,
    'figure.figsize': (9, 6),
    'figure.autolayout': True,
    'font.family': 'serif',
    'lines.linewidth': 1.5,
    'font.size': fontsize,
    'xtick.labelsize': fontsize,
    'ytick.labelsize': fontsize,
    'legend.fontsize': fontsize,
    'legend.loc': 'best',
    'axes.edgecolor': '#b0b0b0',
    'grid.color': '#707070',
    'xtick.color': '#b0b0b0',
    'ytick.color': '#b0b0b0',
    'savefig.dpi': 80,
})

---
## Part 1 — Homodyne Michelson

Standard Michelson with a local oscillator (LO) picked off from the input, phase-shifted by $\phi_\mathrm{hd}$, then beaten against the antisymmetric port field at a homodyne beamsplitter.

The LO field is:
$$E_\mathrm{LO} = -r_\mathrm{LO} e^{-i\phi_\mathrm{HD}} E_\mathrm{in}$$

### 1.1 — Adjacency Matrix

Write down every field in the system, then encode how each one depends on others in matrix $\mathbf{M}$. $M_{ij}$ = contribution of field $j$ to field $i$.

| Symbol | What it is |
|--------|------------|
| $E_1$  | Input field |
| $E_2$  | Field hitting the Michelson BS |
| $E_3$  | X-arm, going toward end mirror |
| $E_4$  | Y-arm, going toward end mirror |
| $E_5$  | Returning from X-arm |
| $E_6$  | Returning from Y-arm |
| $E_7$  | Antisymmetric port field $E_\mathrm{as}$ |
| $E_8$  | Local oscillator $E_\mathrm{lo}$ |
| $E_9$  | Field at PD$_A$ |
| $E_{10}$ | Field at PD$_B$ |

In [ ]:
E1, E2, E3, E4, E5, E6, E7, E8, E9, E10 = sp.symbols('E1:11')

r_bs, t_bs = sp.symbols('r_bs t_bs', positive=True)
r_x,  r_y  = sp.symbols('r_x r_y',   positive=True)
r_m1, t_m1 = sp.symbols('r_m1 t_m1', positive=True)
r_m2       = sp.symbols('r_m2',       positive=True)
r_hd, t_hd = sp.symbols('r_hd t_hd', positive=True)

phi_x, phi_y, phi_hd = sp.symbols('phi_x phi_y phi_hd', real=True)

In [ ]:
M = sp.zeros(10, 10)

In [ ]:
# E2 = t_m1 * E1  (transmitted through input BS)
M[1, 0] = t_m1

In [ ]:
# E3 = t_bs * e^{-i phi_x} * E2  (into X-arm)
M[2, 1] = t_bs * sp.exp(-sp.I * phi_x)

In [ ]:
# E4 = -r_bs * e^{-i phi_y} * E2  (into Y-arm, reflection sign)
M[3, 1] = -r_bs * sp.exp(-sp.I * phi_y)

In [ ]:
# E5 = -r_x * e^{-i phi_x} * E3  (reflected from X end mirror)
M[4, 2] = -r_x * sp.exp(-sp.I * phi_x)

In [ ]:
# E6 = -r_y * e^{-i phi_y} * E4
M[5, 3] = -r_y * sp.exp(-sp.I * phi_y)

In [ ]:
# E7 (AS port) = r_bs*E5 + t_bs*E6
M[6, 4] = r_bs
M[6, 5] = t_bs

In [ ]:
# E8 (LO) = r_m1 * r_m2 * e^{-i phi_hd} * E1
M[7, 0] = r_m1 * r_m2 * sp.exp(-sp.I * phi_hd)

In [ ]:
# E9 (PD_A) = t_hd*E7 + r_hd*E8
M[8, 6] = t_hd
M[8, 7] = r_hd

In [ ]:
# E10 (PD_B) = -r_hd*E7 + t_hd*E8
M[9, 6] = -r_hd
M[9, 7] =  t_hd

In [ ]:
M

### 1.2 — Electric Field Transfer Functions

The system is $\mathbf{E} = \mathbf{M}\cdot\mathbf{E} + \text{source}$, rearranging to $(\mathbf{M} - \mathbf{I})\mathbf{E} = -\text{source}$. Invert $(\mathbf{M}-\mathbf{I})$ to get all transfer functions at once.

$$\frac{E_A}{E_\mathrm{in}} = G_{9,1}, \qquad \frac{E_B}{E_\mathrm{in}} = G_{10,1}$$

In [ ]:
I10 = sp.eye(10)

In [ ]:
# this takes a moment — sympy inverting a 10x10 symbolic matrix
G = (M - I10).inv()

In [ ]:
TF_A = sp.simplify(G[8, 0])
TF_A

In [ ]:
TF_B = sp.simplify(G[9, 0])
TF_B

### 1.3 — Substitutions

Plug in physically motivated simplifications:
- Phase basis: $\phi_x = \phi_c + \phi_d$, $\phi_y = \phi_c - \phi_d$
- Ideal 50:50 Michelson BS: $r_\mathrm{bs} = t_\mathrm{bs} = 1/\sqrt{2}$
- Perfect end mirrors: $r_x = r_y = 1$, perfect steering: $r_{m2} = 1$
- Ideal 50:50 homodyne and input BS

In [ ]:
phi_c, phi_d = sp.symbols('phi_c phi_d', real=True)

In [ ]:
subs = {
    phi_x : phi_c + phi_d,
    phi_y : phi_c - phi_d,
    r_bs  : 1/sp.sqrt(2),
    t_bs  : 1/sp.sqrt(2),
    r_x   : 1,
    r_y   : 1,
    r_m2  : 1,
    r_hd  : 1/sp.sqrt(2),
    t_hd  : 1/sp.sqrt(2),
    r_m1  : 1/sp.sqrt(2),
    t_m1  : 1/sp.sqrt(2),
}

In [ ]:
EA = sp.simplify(TF_A.subs(subs))
EA

In [ ]:
EB = sp.simplify(TF_B.subs(subs))
EB

### 1.4 — Power Transfer Functions

$P = |E|^2$. After factoring and simplifying:

$$P_A = \frac{1}{4}\left[1 + \sin^2(2\phi_d) + 2\sin(2\phi_d)\sin(2\phi_c - \phi_{hd})\right]$$

$$P_B = \frac{1}{4}\left[1 + \sin^2(2\phi_d) - 2\sin(2\phi_d)\sin(2\phi_c - \phi_{hd})\right]$$

$P_\mathrm{diff}$ is your error signal. Setting $\phi_\mathrm{hd} = 2\phi_c - \pi/2$ gives $P_\mathrm{diff}|_\mathrm{opt} = \sin(2\phi_d)$.

In [ ]:
PA_sym = sp.simplify(EA * sp.conjugate(EA))
PA_sym

In [ ]:
PB_sym = sp.simplify(EB * sp.conjugate(EB))
PB_sym

In [ ]:
Psum_sym = sp.simplify(PA_sym + PB_sym)
Psum_sym

In [ ]:
Pdiff_sym = sp.simplify(PA_sym - PB_sym)
Pdiff_sym

In [ ]:
def Pa(pc, pd, phd): return (3 - np.cos(4*pd) + 4*np.sin(2*pd)*np.sin(2*pc - phd)) / 8
def Pb(pc, pd, phd): return (3 - np.cos(4*pd) - 4*np.sin(2*pd)*np.sin(2*pc - phd)) / 8

In [ ]:
phi_d_arr  = np.linspace(-np.pi, np.pi, 1000)
phi_c_arr  = np.linspace(-np.pi, np.pi, 1000)
phi_hd_arr = np.linspace(-np.pi, np.pi, 1000)

### 1.5 — Interpretation

#### Plot 1: Power vs $\phi_d$

Optimal homodyne angle $\phi_\mathrm{hd} = 2\phi_c - \pi/2$, $\phi_c = 0$. $P_A - P_B = \sin(2\phi_d)$ crosses zero at $\phi_d = 0$ — your dark-fringe lock point.

In [ ]:
pc0     = 0.0
phd_opt = 2*pc0 - np.pi/2

pa_ = Pa(pc0, phi_d_arr, phd_opt)
pb_ = Pb(pc0, phi_d_arr, phd_opt)

In [ ]:
fig, ax = plt.subplots()
ax.plot(phi_d_arr, pa_,       lw=2,          label=r'$P_A$')
ax.plot(phi_d_arr, pb_,       lw=2,          label=r'$P_B$')
ax.plot(phi_d_arr, pa_ + pb_, lw=2, ls='--', label=r'$P_A + P_B$')
ax.plot(phi_d_arr, pa_ - pb_, lw=2, ls='-.', label=r'$P_A - P_B$')
ax.axhline(0, color='w', lw=0.7)
ax.axvline(0, color='w', lw=0.7, ls=':')
ax.set_xlabel(r'Differential phase $\phi_d$ [rad]')
ax.set_ylabel('Normalised Power')
ax.set_title(r'Power vs $\phi_d$ — optimal $\phi_{hd}$, $\phi_c = 0$')
ax.set_xticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
ax.set_xticklabels([r'$-\pi$', r'$-\pi/2$', r'$0$', r'$\pi/2$', r'$\pi$'])
ax.legend()
ax.grid()
plt.show()

#### Plot 2: Power vs $\phi_c$

$\phi_d = \pi/4$, $\phi_\mathrm{hd} = 0$. Common phase shuffles power between detectors but the sum stays flat.

In [ ]:
phi_d_bright = np.pi / 4

pa2_ = Pa(phi_c_arr, phi_d_bright, 0.0)
pb2_ = Pb(phi_c_arr, phi_d_bright, 0.0)

In [ ]:
fig, ax = plt.subplots()
ax.plot(phi_c_arr, pa2_,        lw=2,          label=r'$P_A$')
ax.plot(phi_c_arr, pb2_,        lw=2,          label=r'$P_B$')
ax.plot(phi_c_arr, pa2_ + pb2_, lw=2, ls='--', label=r'$P_A + P_B$')
ax.plot(phi_c_arr, pa2_ - pb2_, lw=2, ls='-.', label=r'$P_A - P_B$')
ax.axhline(0, color='w', lw=0.7)
ax.set_xlabel(r'Common phase $\phi_c$ [rad]')
ax.set_ylabel('Normalised Power')
ax.set_title(r'Power vs $\phi_c$ — $\phi_d = \pi/4$, $\phi_{hd} = 0$')
ax.set_xticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
ax.set_xticklabels([r'$-\pi$', r'$-\pi/2$', r'$0$', r'$\pi/2$', r'$\pi$'])
ax.legend()
ax.grid()
plt.show()

#### Plot 3: Power vs $\phi_\mathrm{hd}$

$\phi_d = \pi/4$, $\phi_c = 0$. Sweeping $\phi_\mathrm{hd}$ rotates the interference between $E_\mathrm{as}$ and $E_\mathrm{lo}$.

Catch: only works on a bright fringe. On the dark fringe $E_\mathrm{as} = 0$ and the LO has nothing to beat against — this is why real experiments need a CLF.

In [ ]:
pa3_ = Pa(0.0, phi_d_bright, phi_hd_arr)
pb3_ = Pb(0.0, phi_d_bright, phi_hd_arr)

In [ ]:
fig, ax = plt.subplots()
ax.plot(phi_hd_arr, pa3_,        lw=2,          label=r'$P_A$')
ax.plot(phi_hd_arr, pb3_,        lw=2,          label=r'$P_B$')
ax.plot(phi_hd_arr, pa3_ + pb3_, lw=2, ls='--', label=r'$P_A + P_B$')
ax.plot(phi_hd_arr, pa3_ - pb3_, lw=2, ls='-.', label=r'$P_A - P_B$')
ax.axhline(0, color='w', lw=0.7)
ax.set_xlabel(r'Homodyne angle $\phi_{hd}$ [rad]')
ax.set_ylabel('Normalised Power')
ax.set_title(r'Power vs $\phi_{hd}$ — $\phi_d = \pi/4$, $\phi_c = 0$')
ax.set_xticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
ax.set_xticklabels([r'$-\pi$', r'$-\pi/2$', r'$0$', r'$\pi/2$', r'$\pi$'])
ax.legend()
ax.grid()
plt.show()

---
## Part 2 — Asymmetric Michelson

Now $L_x \gg L_y$. Sidebands from mirror motion pick up different phase delays in the two arms because $k = \Omega/c$ and each frequency sees a different $\Omega$. So the Michelson response becomes frequency-dependent.

Goal: $P_\mathrm{as}/\phi_\mathrm{CARM}(\omega)$ and $P_\mathrm{as}/\phi_\mathrm{DARM}(\omega)$.

### 2.1 — Field Transfer Functions

One-bounce Michelson. Derive $E_x/E_\mathrm{in}$, $E_y/E_\mathrm{in}$, $E_\mathrm{as}/E_\mathrm{in}$.

In [ ]:
r_bs2, t_bs2 = sp.symbols('r_bs t_bs', positive=True)
r_x2, r_y2   = sp.symbols('r_x r_y',   positive=True)
phi_x2, phi_y2 = sp.symbols('phi_x phi_y', real=True)

Ex_over_Ein  = -r_x2 * t_bs2 * sp.exp(-sp.I * phi_x2)
Ey_over_Ein  =  r_y2 * r_bs2 * sp.exp(-sp.I * phi_y2)
Eas_over_Ein = (-r_x2 * r_bs2 * t_bs2 * sp.exp(-sp.I * 2*phi_x2)
               + r_y2 * r_bs2 * t_bs2 * sp.exp(-sp.I * 2*phi_y2))

print('E_x / E_in  =', Ex_over_Ein)
print('E_y / E_in  =', Ey_over_Ein)
print('E_as / E_in =', Eas_over_Ein)

### 2.2 — End Mirror Modulation

Apply CARM modulation: $\phi_x(t) = \phi_y(t) = \Gamma\cos(\omega t)$

Expand $e^{-i\Gamma\cos(\omega t)} \approx 1 + \frac{i\Gamma}{2}e^{i\omega t} + \frac{i\Gamma}{2}e^{-i\omega t}$

This gives three frequency components: $\omega_0$, $\omega_0 \pm \omega$. All three beams share the same DC arm phase at the end mirror (the mirror moved while the carrier was already propagating, so all see the same DC delay).

In [ ]:
# E_x(t) = -E0 r_x t_bs e^{-i phi_x0} [ e^{i w0 t}
#             + i*Gamma/2 * e^{i(w0+w)t}
#             + i*Gamma/2 * e^{i(w0-w)t} ]
#
# E_y(t) = E0 r_y r_bs e^{-i phi_y0} [ e^{i w0 t}
#             + i*Gamma/2 * e^{i(w0+w)t}
#             + i*Gamma/2 * e^{i(w0-w)t} ]
print('Three sidebands per arm: carrier + USB + LSB')

### 2.3 — Propagate to the AS Port

Each field propagates back with $e^{-ikL}$ where $k = \Omega/c$. The carrier, USB, and LSB all have different $\Omega$, so they accrue different return phases:

- Carrier: $\phi_{x0} = \omega_0 L_x / c$
- Upper sideband: $\phi_{x,\mathrm{usb}} = \phi_{x0}(1 + \omega/\omega_0)$
- Lower sideband: $\phi_{x,\mathrm{lsb}} = \phi_{x0}(1 - \omega/\omega_0)$

In [ ]:
# After combining at the BS (ideal 50:50, balanced arms r_x = r_y)
# and switching to phi_c0, phi_d0 basis:
#
# Carrier:  E_as,0   = i E0 r_x e^{i w0 t} e^{-i 2 phi_c0} sin(2 phi_d0)
#
# Upper SB: E_as,usb = -(Gamma/2) E0 r_x e^{i(w0+w)t}
#                       * e^{-i(2+w/w0)phi_c0} * sin((2+w/w0) phi_d0)
#
# Lower SB: E_as,lsb = -(Gamma/2) E0 r_x e^{i(w0-w)t}
#                       * e^{-i(2-w/w0)phi_c0} * sin((2-w/w0) phi_d0)
print('E_as = carrier + USB + LSB')

### 2.4 — Power Response to CARM Motion

$P_\mathrm{as} = |E_\mathrm{as}|^2$. Carrier $\times$ USB cross term oscillates at $+\omega$, carrier $\times$ LSB at $-\omega$. Combined they give a $\sin(\omega t)$ modulation. Drop $\Gamma^2 \approx 0$.

$$\frac{P_\mathrm{as}}{P_\mathrm{in}} = r_x^2\left[\sin^2(2\phi_d) - 2\Gamma\sin(4\phi_d)\sin\!\left(\frac{\omega\phi_d}{\omega_0}\right)\sin\!\left(\omega t + \frac{\omega\phi_c}{\omega_0}\right)\right]$$

In [ ]:
# cross terms: E_as,0 * conj(E_as,usb) + E_as,0 * conj(E_as,lsb) + c.c.
# => -2 Gamma sin(4 phi_d) sin(w phi_d/w0) sin(wt + w phi_c/w0)
print('P_as / P_in confirmed')

### 2.5 — Demodulation

Integrate over one cycle of $\omega t$ multiplied by $1$, $\cos(\omega t)$, $\sin(\omega t)$.

In [ ]:
# DC: sin(wt + A) integrates to zero over a full cycle
# => P_as^DC = r_x^2 sin^2(2 phi_d)   (same as static homodyne — sanity check)
print('P_as^DC = r_x^2 * sin^2(2 phi_d)')

In [ ]:
# I quadrature: sin(wt + A) * cos(wt) = 1/2 sin(2wt+A) + 1/2 sin(A)
# sin(2wt) integrates to zero, sin(A) = sin(w phi_c/w0) is a constant
# but our signal is proportional to sin(wt), which is orthogonal to cos(wt)
# => P_as^I = 0
print('P_as^I = 0')

In [ ]:
# Q quadrature: sin(wt + A) * sin(wt) = 1/2 cos(A) - 1/2 cos(2wt+A)
# cos(2wt) integrates to zero, leaving (1/2) cos(A) = (1/2) cos(w phi_c/w0)
# => P_as^Q = -r_x^2 Gamma sin(4 phi_d) sin(w phi_d/w0) cos(w phi_c/w0)
print('P_as^Q = -r_x^2 * Gamma * sin(4 phi_d) * sin(w phi_d/w0) * cos(w phi_c/w0)')

### 2.6 — CARM Frequency Response

$P_\mathrm{as}(\omega) = P_\mathrm{as}^I + i P_\mathrm{as}^Q$. Since $P_\mathrm{as}^I = 0$:

$$\frac{P_\mathrm{as}}{\Gamma}(\omega) = -i r_x^2 \sin(4\phi_d)\sin\!\left(\frac{\omega\phi_d}{\omega_0}\right)\cos\!\left(\frac{\omega\phi_c}{\omega_0}\right)$$

Parameters: $r_x = 1$, $L_x = 1000\,\mathrm{m}$, $L_y = 1\,\mathrm{m}$, $\lambda = 1064\,\mathrm{nm}$

In [ ]:
c      = 3e8
lam    = 1064e-9
omega0 = 2*np.pi * c / lam

In [ ]:
Lx = 1000.0
Ly = 1.0
rx = 1.0

In [ ]:
phi_x0_val = omega0 * Lx / c
phi_y0_val = omega0 * Ly / c

phi_c0_val = (phi_x0_val + phi_y0_val) / 2
phi_d0_val = (phi_x0_val - phi_y0_val) / 2

print(f'phi_x0 = {phi_x0_val:.4e} rad')
print(f'phi_y0 = {phi_y0_val:.4e} rad')
print(f'phi_c0 = {phi_c0_val:.4e} rad')
print(f'phi_d0 = {phi_d0_val:.4e} rad')

In [ ]:
f_arr     = np.logspace(0, 6, 2000)
omega_arr = 2 * np.pi * f_arr

In [ ]:
TF_CARM = (-1j * rx**2
           * np.sin(4 * phi_d0_val)
           * np.sin(omega_arr * phi_d0_val / omega0)
           * np.cos(omega_arr * phi_c0_val / omega0))

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(9, 8))

ax1.loglog(f_arr, np.abs(TF_CARM), color='steelblue', lw=2)
ax1.set_ylabel(r'$|P_\mathrm{as}/\Gamma|$')
ax1.set_title(r'CARM Frequency Response — Asymmetric Michelson')
ax1.grid(which='both', alpha=0.4)

In [ ]:
ax2.semilogx(f_arr, np.angle(TF_CARM, deg=True), color='tomato', lw=2)
ax2.set_ylabel(r'Phase [deg]')
ax2.set_xlabel(r'Frequency [Hz]')
ax2.set_yticks([-180, -90, 0, 90, 180])
ax2.grid(which='both', alpha=0.4)

plt.tight_layout()
plt.show()

f_null = omega0 * np.pi / (2 * np.pi * phi_d0_val)
print(f'First null ~ {f_null:.2f} Hz')